In [ ]:
#%% md
# Ноутбук 2: Сравнение эмбеддинг-пайплайнов для дедупликации
#
# Совмещены три гипотезы про ОДНУ ось — "чем векторизовать текст для поиска дублей":
#
#   Гипотеза D (лёгкие статические эмбеддинги) — Model2Vec вместо тяжёлого трансформера
#       для быстрого первого прохода.
#   Гипотеза E (иерархический coarse-to-fine поиск) — грубый поиск кандидатов по укороченному
#       вектору (Matryoshka-подобная идея), затем точная проверка на полном векторе.
#   Гипотеза F (сравнение моделей эмбеддингов) — RuModernBERT-small vs BGE-M3
#       (мультиязычная multi-granularity модель) на одном и том же наборе новостей.
#
# Все три сравниваются друг с другом и с TF-IDF-baseline по ЕДИНОЙ метрике качества и скорости,
# чтобы результаты были сопоставимы. НЕ включаем сюда экономический анализ (ноутбук 1),
# LLM-валидацию (ноутбук 3) и потоковую обработку (ноутбук 4) — здесь только сравнение векторизации.
#%%


In [ ]:
from pathlib import Path
import re, time, tracemalloc
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('embedding_comparison_results')
OUT_DIR.mkdir(exist_ok=True)
WINDOW_DAYS = 14

df = pd.read_csv(DATA_PATH, parse_dates=['post_dttm']).reset_index(drop=True)

def normalize_text(text):
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df['text_normalized'] = df['data'].fillna('').map(normalize_text)
df = df[df.text_normalized.str.len().gt(0)].copy().reset_index(drop=True)
print(f'Новостей: {len(df):,}')

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.parent[b] = a

def clusters_from_edges(n, edges):
    uf = UnionFind(n)
    for a, b in edges: uf.union(a, b)
    return pd.factorize([uf.find(i) for i in range(n)])[0]

def top_k_edges(vectors, dates, threshold, n_neighbors=6, batch_size=128):
    ts = pd.to_datetime(dates, utc=True).astype('int64').to_numpy()
    max_delta = pd.Timedelta(days=WINDOW_DAYS).value
    edges = []
    for start in range(0, len(ts), batch_size):
        stop = min(start + batch_size, len(ts))
        sim = vectors[start:stop] @ vectors.T
        if hasattr(sim, 'toarray'): sim = sim.toarray()
        idx = np.argpartition(sim, -n_neighbors, axis=1)[:, -n_neighbors:]
        for local_i, candidates in enumerate(idx):
            i = start + local_i
            for j in candidates:
                if i < j and sim[local_i, j] >= threshold and abs(ts[i] - ts[j]) <= max_delta:
                    edges.append((i, int(j)))
    return edges

REFERENCE_THRESHOLD = 0.90
tfidf_ref = TfidfVectorizer(analyzer='word', ngram_range=(2, 5), min_df=4, max_features=500_000)
tfidf_ref_vectors = tfidf_ref.fit_transform(df.text_normalized)
reference_clusters = clusters_from_edges(len(df), top_k_edges(tfidf_ref_vectors, df.post_dttm, REFERENCE_THRESHOLD))
n_ref_dup = (pd.Series(reference_clusters).value_counts() > 1).sum()
print(f'Референсных кластеров дублей (TF-IDF {REFERENCE_THRESHOLD}): {n_ref_dup}')

In [ ]:
#%% md
## Единая функция оценки для всех эмбеддинг-пайплайнов
# Как оцениваем результат КАЖДОЙ гипотезы (D, E, F) — по четырём осям:
#  1. Согласие с референсным разбиением TF-IDF (Adjusted Rand Index, Normalized Mutual Info) —
#     насколько похожие кластеры дублей находит метод.
#  2. Время вычисления эмбеддингов + поиска дублей (сек).
#  3. Пиковое потребление памяти во время эмбеддинга (МБ, через tracemalloc).
#  4. Число найденных пар-дублей и число кластеров — для понимания "агрессивности" метода.
#%%
results_table = []

def evaluate_method(name, embed_fn, threshold, n_neighbors=6):
    tracemalloc.start()
    t0 = time.time()
    vectors = embed_fn(df.text_normalized.tolist())
    embed_time = time.time() - t0
    edges = top_k_edges(vectors, df.post_dttm, threshold, n_neighbors=n_neighbors)
    clusters = clusters_from_edges(len(df), edges)
    total_time = time.time() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    ari = adjusted_rand_score(reference_clusters, clusters)
    nmi = normalized_mutual_info_score(reference_clusters, clusters)
    sizes = pd.Series(clusters).value_counts()

    row = dict(method=name, embed_time_sec=round(embed_time, 1), total_time_sec=round(total_time, 1),
               peak_memory_mb=round(peak / 1e6, 1), pairs_found=len(edges),
               duplicate_clusters=int((sizes > 1).sum()), ari_vs_tfidf=round(ari, 3), nmi_vs_tfidf=round(nmi, 3))
    results_table.append(row)
    print(row)
    return clusters, vectors

In [ ]:
#%% md
### Гипотеза D: лёгкие статические эмбеддинги (Model2Vec) как быстрый baseline
# Что проверяем: даёт ли лёгкая статическая модель (без forward pass трансформера)
#     приемлемое качество дедупликации за заметно меньшее время — как первый грубый проход
#     перед дорогим ModernBERT.
# Как применяем: используем предобученную мультиязычную модель Model2Vec, считаем
#     эмбеддинги, ищем дубли тем же top_k_edges на тех же условиях (окно, порог), что и
#     для остальных методов.
# Как оцениваем результат: через evaluate_method() — ARI/NMI к TF-IDF-референсу, время, память.
#%%
try:
    from model2vec import StaticModel
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'model2vec'])
    from model2vec import StaticModel

try:
    m2v_model = StaticModel.from_pretrained('minishlab/potion-multilingual-128M')
    def embed_model2vec(texts):
        return m2v_model.encode(texts, show_progress_bar=True)
    clusters_m2v, _ = evaluate_method('Model2Vec (лёгкий static)', embed_model2vec, threshold=0.80)
except Exception as exc:
    print(f'Model2Vec недоступен ({exc}) — пропускаем эту гипотезу в этом запуске.')
    clusters_m2v = None

In [ ]:
#%% md
### Гипотеза F: сравнение эмбеддинг-моделей (RuModernBERT-small vs BGE-M3)
# Что проверяем: какая модель эмбеддингов лучше подходит для семантической дедупликации
#     русскоязычных новостей — компактная RuModernBERT-small или мультиязычная multi-granularity
#     BGE-M3 (обычно сильнее на semantic similarity задачах, но тяжелее).
# Как применяем: считаем полноразмерные эмбеддинги обеими моделями на одном тексте,
#     ищем дубли тем же top_k_edges с одинаковым порогом/окном.
# Как оцениваем результат: через evaluate_method() — те же 4 метрики, что у гипотезы D,
#     плюс прямое сравнение ARI/NMI RuModernBERT vs BGE-M3 друг с другом (не только к TF-IDF).
#%%
import torch
from transformers import AutoModel, AutoTokenizer

def transformer_embed_fn(model_name, max_length=512, batch_size=64):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    def embed(texts):
        result = []
        with torch.inference_mode():
            for start in tqdm(range(0, len(texts), batch_size), desc=f'Эмбеддинги: {model_name}'):
                batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                                  max_length=max_length, return_tensors='pt').to(device)
                hidden = model(**batch).last_hidden_state
                mask = batch['attention_mask'].unsqueeze(-1)
                pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
                result.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy())
        return np.vstack(result)
    return embed

modernbert_embed = transformer_embed_fn('deepvk/RuModernBERT-small', max_length=1024)
clusters_modernbert, modernbert_vectors = evaluate_method(
    'RuModernBERT-small', modernbert_embed, threshold=0.93)

try:
    bge_embed = transformer_embed_fn('BAAI/bge-m3', max_length=1024)
    clusters_bge, _ = evaluate_method('BGE-M3', bge_embed, threshold=0.93)
    ari_cross = adjusted_rand_score(clusters_modernbert, clusters_bge)
    print(f'ARI между RuModernBERT-small и BGE-M3 напрямую: {ari_cross:.3f}')
except Exception as exc:
    print(f'BGE-M3 недоступен в этом окружении ({exc}) — пропускаем.')

In [ ]:
#%% md
### Гипотеза E: иерархический coarse-to-fine поиск (Matryoshka-подобный)
# Что проверяем: можно ли ускорить поиск дублей, сначала грубо отобрав кандидатов по
#     укороченному вектору (первые K измерений эмбеддинга), а затем точно проверив только
#     кандидатов на полном векторе — вместо полного top-k поиска сразу по 100% измерений.
# ВАЖНО: RuModernBERT-small не обучалась с Matryoshka-loss, поэтому усечение вектора —
#     лишь ПРИБЛИЖЕНИЕ идеи (в отличие от специально обученных Matryoshka-моделей, где
#     усечённые векторы гарантированно остаются осмысленными). Это ограничение явно
#     фиксируем в выводах.
# Как применяем: берём уже посчитанные (в гипотезе F) modernbert_vectors, для грубого прохода
#     используем первые COARSE_DIM измерений с более низким порогом (шире невод), для точного —
#     полный вектор с обычным порогом, но только среди пар-кандидатов грубого прохода.
# Как оцениваем результат: сравниваем итоговое множество пар-дублей и время с полным
#     top_k_edges по 100% измерений — ожидаем близкое качество (высокий ARI к полному
#     проходу) при меньшем времени точной проверки.
#%%
COARSE_DIM = 128 if modernbert_vectors.shape[1] > 128 else modernbert_vectors.shape[1] // 2
COARSE_THRESHOLD = 0.75
FINE_THRESHOLD = 0.93

t0 = time.time()
coarse_vectors = modernbert_vectors[:, :COARSE_DIM]
norms = np.clip(np.linalg.norm(coarse_vectors, axis=1, keepdims=True), 1e-9, None)
coarse_vectors = coarse_vectors / norms
coarse_edges = top_k_edges(coarse_vectors, df.post_dttm, COARSE_THRESHOLD, n_neighbors=15)
coarse_time = time.time() - t0

t1 = time.time()
fine_edges = [
    (i, j) for (i, j) in coarse_edges
    if float(modernbert_vectors[i] @ modernbert_vectors[j]) >= FINE_THRESHOLD
]
fine_time = time.time() - t1

clusters_hier = clusters_from_edges(len(df), fine_edges)
total_hier_time = coarse_time + fine_time

ari_hier = adjusted_rand_score(reference_clusters, clusters_hier)
nmi_hier = normalized_mutual_info_score(reference_clusters, clusters_hier)
ari_vs_full = adjusted_rand_score(clusters_modernbert, clusters_hier)

print(f'Coarse-этап: {len(coarse_edges)} пар-кандидатов за {coarse_time:.1f} с')
print(f'Fine-этап: {len(fine_edges)} подтверждённых пар за {fine_time:.1f} с')
print(f'Итого: {total_hier_time:.1f} с; ARI к TF-IDF-референсу={ari_hier:.3f}; '
      f'ARI к полному ModernBERT-проходу={ari_vs_full:.3f}')

results_table.append(dict(method='ModernBERT coarse-to-fine (Matryoshka-подобный)',
                           embed_time_sec=0.0, total_time_sec=round(total_hier_time, 1),
                           peak_memory_mb=np.nan, pairs_found=len(fine_edges),
                           duplicate_clusters=int((pd.Series(clusters_hier).value_counts() > 1).sum()),
                           ari_vs_tfidf=round(ari_hier, 3), nmi_vs_tfidf=round(nmi_hier, 3)))

In [ ]:
#%% md
## Сводная таблица по всем гипотезам (D, E, F)
#%%
report = pd.DataFrame(results_table).sort_values('total_time_sec')
display(report)
report.to_csv(OUT_DIR / 'embedding_methods_comparison.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(report['total_time_sec'], report['ari_vs_tfidf'], s=80)
for _, row in report.iterrows():
    ax.annotate(row['method'], (row['total_time_sec'], row['ari_vs_tfidf']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_xlabel('Общее время (сек)')
ax.set_ylabel('ARI к TF-IDF-референсу (качество)')
ax.set_title('Компромисс скорость / качество по методам эмбеддинга')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'speed_quality_tradeoff.png', dpi=150)
plt.show()